<a href="https://colab.research.google.com/github/kph4br/ds2002-fa26/blob/main/studio/2026-10-14-Mini-ETL-on-Walmart-Sample-Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Mini-ETL on Walmart Sample

**Studio — 2026-10-14 · Fall 2026**  
**Class time:** 45 minutes

---

## The midterm, in miniature

Twelve rows of the real thing. Every problem in this sample is in the full dataset you get Monday, just at a scale where you can still see the answer.

You are writing the whole pipeline as **functions**, not as loose cells. That is not stylistic — Monday you will run the same logic on a much larger file, and a pipeline built from functions is one you can re-run and hand to a teammate. A pipeline built from cells in a specific click order is not.

The extract stage is worked below as an example. Transform and load are yours to build.

In [1]:
import pandas as pd, sqlite3
from io import StringIO

RAW = '''transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1,3.86
TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1,5.00
TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2,$4.11
TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3,$4.11
TXN5,fl-239,2024-09-06T12:15:00,WATER-24,bottled water,grocery,4,$5.00
TXN6,FL-105,2024-09-07 08:02:00,BTL-WATER-24,Bottled Water 24pk,Grocery,6,5.00
TXN7,FL-105,2024-09-07T09:45:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,5,4.11
TXN8,FL-239,,PT-12,pop-tarts strawberry,Breakfast,2,$4.11
TXN9,FL-105,2024-09-07 14:20:00,FLASH-AA,Flashlight AA,Hardware,NULL,12.99
TXN10,FL-239,2024-09-07T15:00:00,POPTART-BLUE,Pop-Tarts Blueberry,Breakfast,1,4.11
TXN11,FL-105,09/07/2024 16:10,SKU#459812,Strawberry Pop-Tart,breakfast,4,4.11'''

### Extract — worked example

The extract function does one job: get the raw data in, and report what arrived. It does **not** clean anything. Keeping the stages separate is what lets you answer "was this a source problem or did I do it?" later.

In [2]:
def extract():
    """Read the raw transactions. No cleaning here -- report only."""
    df = pd.read_csv(StringIO(RAW))
    print(f'extracted {len(df)} rows, {len(df.columns)} columns')
    print('dtypes that arrived as text:',
          [c for c in df.columns if df[c].dtype == object])
    print('nulls:', df.isnull().sum().sum())
    print('exact duplicates:', df.duplicated().sum())
    return df

raw = extract()
raw.head()

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
1,TXN2,fl-105,09/06/2024 10:40,PT-12,pop-tarts strawberry,breakfast,1.0,3.86
2,TXN3,FL-239,2024-09-06T11:00:00,WATER-24,Bottled Water,Grocery,-1.0,5.00
3,TXN1,FL-105,2024-09-06 10:06:00,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,2.0,$4.11
4,TXN4,FL 239,09/06/2024 11:30,SKU#459812,Strawberry Pop-Tart,Breakfast,3.0,$4.11


Note what the report already told us: `quantity` and `unit_price` both arrived as text, there is a null, and there is a duplicate. That is your transform to-do list, generated from the data instead of from guessing.

**TODO:** before writing any transform, list the specific problems you can see in these twelve rows. There are at least six.

1. TXN1 shows up twice (duplicate row)
2. unit_price has a dollar sign on some rows, so it is read in as text instead of a number
3. TXN9's quantity is NULL (pandas reads it as missing automatically)
4. TXN3's quantity is -1, which can't be a normal sale
5. store_id is spelled 4 different ways for the same 2 stores (FL-105, fl-105, FL 239, fl-239)
6. timestamp comes in 3 different formats, and TXN8's is blank
7. The same product has different SKUs: 3 for strawberry Pop-Tarts and 2 for the bottled water
8. product_name and category have inconsistent capitalization (Breakfast vs breakfast)

### Build 1 — Transform: types and duplicates

**TODO:** write `transform_types(df)` that returns a new frame with:

- exact duplicate rows removed
- `unit_price` as a float
- `quantity` numeric, with a stated decision about the null and the negative
- `store_id` normalized — note the four spellings of the same two stores
- `timestamp` as a real datetime, coercing the empty one

It must not modify the frame passed in. Print a before/after row count.

In [3]:
def transform_types(df):
    out = df.copy()
    # TODO: dedupe
    # TODO: unit_price -> float
    # TODO: quantity -> numeric, then your decision on null / negative
    # TODO: store_id -> one consistent format
    # TODO: timestamp -> datetime, errors='coerce', format='mixed'
    out = out.drop_duplicates()
    out['unit_price'] = out['unit_price'].str.replace('$', '').astype(float)
    out['quantity'] = pd.to_numeric(out['quantity'], errors='coerce')
    out['store_id'] = out['store_id'].str.upper().str.replace(' ', '-')
    out['timestamp'] = pd.to_datetime(out['timestamp'], errors='coerce', format='mixed')
    # decision: a missing quantity stays missing (not 0), but a negative
    # quantity is a return, not a sale, so that row gets dropped
    out = out[out['quantity'].isna() | (out['quantity'] >= 0)]
    return out

typed = transform_types(raw)
print('rows:', len(raw), '->', len(typed))
typed.dtypes

rows: 12 -> 10


,0
transaction_id,object
store_id,object
timestamp,datetime64[ns]
sku,object
product_name,object
category,object
quantity,float64
unit_price,float64


I went from 12 rows to 10. One was the TXN1 duplicate and the other was TXN3 with quantity -1. I dropped the -1 because a negative quantity is a return, not a sale, and keeping it would take units away from the sales numbers. I left TXN9's missing quantity as missing instead of making it 0, since we don't know how many were sold. I stripped the dollar sign so unit_price could be a float, made store IDs uppercase with a dash so there's only FL-105 and FL-239, and turned timestamp into datetime.

### Build 2 — Transform: consolidate the products

Four SKUs are strawberry Pop-Tarts. One is blueberry — leave it alone, and be ready to say why. Two are the same bottled water.

**TODO:** write `consolidate(df)` that adds a `canonical_sku` column. Any SKU not in your mapping should keep its original value, not become `NaN`.

Then print units per canonical SKU and confirm the total quantity did not change — consolidation renames things, it never adds or removes units.

In [4]:
CANON = {
    # TODO: map the strawberry Pop-Tart SKUs to one value
    # TODO: map the two bottled water SKUs to one value
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'BTL-WATER-24': 'WATER-24',
}

def consolidate(df):
    out = df.copy()
    # TODO: out['canonical_sku'] = ...  (keep unmapped SKUs as themselves)
    out['canonical_sku'] = out['sku'].replace(CANON)
    return out

consolidated = consolidate(typed)
# TODO: assert total quantity is unchanged from `typed`
# TODO: print units per canonical_sku
assert consolidated['quantity'].sum() == typed['quantity'].sum()
print(consolidated.groupby('canonical_sku')['quantity'].sum(min_count=1))

canonical_sku
FLASH-AA          NaN
POPTART-BLUE      1.0
POPTART-STRAW    17.0
WATER-24         10.0
Name: quantity, dtype: float64


**Why blueberry stays separate:** Blueberry is a different flavor, so it's a different product, even though it's the same brand and price. If I merged it in, strawberry would get credit for a sale it didn't make and the strawberry numbers would be wrong.

### Build 3 — Transform: derive what the analysis needs

**TODO:** add `revenue` (quantity times unit price) and a `date` column from the timestamp. The `date` column is what you will join weather onto next week, so it needs to be a date — not a full timestamp, not a string.

In [5]:
def derive(df):
    out = df.copy()
    # TODO: revenue
    # TODO: date (hint: .dt.date, or .dt.normalize() to keep it a datetime)
    out['revenue'] = out['quantity'] * out['unit_price']
    out['date'] = out['timestamp'].dt.normalize()
    return out

enriched = derive(consolidated)
enriched[['transaction_id', 'canonical_sku', 'quantity', 'revenue', 'date']]

,transaction_id,canonical_sku,quantity,revenue,date
0,TXN1,POPTART-STRAW,2.0,8.22,2024-09-06
1,TXN2,POPTART-STRAW,1.0,3.86,2024-09-06
4,TXN4,POPTART-STRAW,3.0,12.33,2024-09-06
5,TXN5,WATER-24,4.0,20.00,2024-09-06
6,TXN6,WATER-24,6.0,30.00,2024-09-07
7,TXN7,POPTART-STRAW,5.0,20.55,2024-09-07
8,TXN8,POPTART-STRAW,2.0,8.22,NaT
9,TXN9,FLASH-AA,NaN,NaN,2024-09-07
10,TXN10,POPTART-BLUE,1.0,4.11,2024-09-07
11,TXN11,POPTART-STRAW,4.0,16.44,2024-09-07


### Build 4 — Load, and verify the load

**TODO:** write `load(df, conn)` that writes the clean frame to a `clean_sales` table, then reads back the row count and asserts it matches. Writing without verifying is how you discover at 11pm that `if_exists='append'` doubled your table.

In [6]:
def load(df, conn, table='clean_sales'):
    # TODO: df.to_sql(...) with index=False and if_exists='replace'
    # TODO: read back COUNT(*) and assert it equals len(df)
    # TODO: print confirmation
    df.to_sql(table, conn, index=False, if_exists='replace')
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    assert count == len(df), f'table has {count} rows, expected {len(df)}'
    print(f'loaded {count} rows into {table}')

conn = sqlite3.connect(':memory:')
load(enriched, conn)

loaded 10 rows into clean_sales


### Build 5 — run the whole thing as one pipeline

**TODO:** wire the stages together in one function, then run it twice against a fresh connection and confirm you get identical results. That is the idempotency property from Monday, tested rather than assumed.

In [7]:
def run_pipeline(conn):
    # TODO: extract -> transform_types -> consolidate -> derive -> load
    # TODO: return the final frame
    df = extract()
    df = transform_types(df)
    df = consolidate(df)
    df = derive(df)
    load(df, conn)
    return df

first = run_pipeline(sqlite3.connect(':memory:'))
second = run_pipeline(sqlite3.connect(':memory:'))

# TODO: assert the two runs produced the same shape and the same revenue total
same_result = (first.shape == second.shape) and (first['revenue'].sum() == second['revenue'].sum())
assert same_result, 'the two runs did not match'
print('shape:', first.shape)
print('revenue total:', round(first['revenue'].sum(), 2))
print('same twice:', same_result)

extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
loaded 10 rows into clean_sales
extracted 12 rows, 8 columns
dtypes that arrived as text: ['transaction_id', 'store_id', 'timestamp', 'sku', 'product_name', 'category', 'unit_price']
nulls: 2
exact duplicates: 1
loaded 10 rows into clean_sales
shape: (10, 11)
revenue total: 123.73
same twice: True


### Build 6 — the question the pipeline exists to answer

**TODO:** did strawberry Pop-Tart sales rise from Sept 6 to Sept 7? Print units by date for your canonical strawberry SKU, and say whether two days is enough evidence to conclude anything.

That second part matters. On the midterm you will have a real baseline to compare against; here you do not, and saying so is the correct answer.

In [8]:
# TODO
straw = enriched[enriched['canonical_sku'] == 'POPTART-STRAW']
by_date = straw.groupby('date')['quantity'].sum()
print(by_date)

no_date = straw.loc[straw['date'].isna(), 'quantity'].sum()
print(f"Strawberry Pop-Tarts went from {by_date.iloc[0]:.0f} units on {by_date.index[0].strftime('%b %d')} "
      f"to {by_date.iloc[1]:.0f} units on {by_date.index[1].strftime('%b %d')}.")
print(f'{no_date:.0f} more strawberry units have no timestamp, so they are not in either day.')

date
2024-09-06    6.0
2024-09-07    9.0
Name: quantity, dtype: float64
Strawberry Pop-Tarts went from 6 units on Sep 06 to 9 units on Sep 07.
2 more strawberry units have no timestamp, so they are not in either day.


Yes, strawberry Pop-Tart sales went up from 6 units on Sept 6 to 9 units on Sept 7. However, two days isn't enough evidence to conclude anything, because we don't have a baseline to compare it to. Without a baseline we can't tell if that's an actual increase or just a normal difference between days.

---

## Checkpoint (participation)

Report your row count after transform, your canonical SKU count, and whether running the pipeline twice gave the same revenue total.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [9]:
# Checkpoint
rows_after_transform = len(typed)   # TODO
canonical_skus = consolidated['canonical_sku'].nunique()         # TODO: how many distinct products you ended with
idempotent = same_result             # TODO: True/False from Build 5
blueberry_call = "it's a different flavor so it's a different product, merging it would wrongfully inflate strawberry sales"       # TODO: why you kept blueberry separate

print('rows after transform:', rows_after_transform)
print('canonical SKUs:', canonical_skus)
print('same result twice:', idempotent)
print('blueberry:', blueberry_call)

rows after transform: 10
canonical SKUs: 4
same result twice: True
blueberry: it's a different flavor so it's a different product, merging it would wrongfully inflate strawberry sales
